https://yt.yandex-team.ru/hahn/navigation?path=//home/ads/users/shevkunov/music_rbe_transformer_tracks

https://yt.yandex-team.ru/hahn/navigation?path=//home/ads/users/shevkunov/music_rbe_users_transformer-2024-07-31

In [1]:
import pickle
import numpy as np

In [2]:
with open("music_rbe__track_wl__v1.pickle", "rb") as f:
    track_wl = pickle.load(f)
    

track_wl_map = dict()
for track_i in track_wl:
    track_wl_map[track_i] = len(track_wl_map)
    
len(track_wl), len(track_wl_map)

(8950, 8950)

In [3]:
with open("music_rbe__tracks_emb__v1.pickle", "rb") as f:
    tracks_emb = pickle.load(f)

len(tracks_emb)

8950

In [4]:
with open("music_rbe__nomm_tracks_emb__v1.pickle", "rb") as f:
    nomm_tracks_emb = pickle.load(f)

len(nomm_tracks_emb)

8950

In [5]:
with open("music_rbe__user2track2score__v1.pickle", "rb") as f:
    user2track2score = pickle.load(f)

In [6]:
user2track2score_clean = dict()
for user_id, tracks in user2track2score.items():
    if len(track_wl.intersection(set(tracks.keys()))) >= 0.5 * len(track_wl):
        user2track2score_clean[user_id] = tracks
    
len(user2track2score_clean)

10179

In [7]:
with open("music_rbe__users_emb_v1.pickle", "rb") as f:
    users_emb = pickle.load(f)
    
len(users_emb)

10179

In [8]:
with open("music_rbe__nomm_users_emb_v1.pickle", "rb") as f:
    nomm_users_emb = pickle.load(f)
    
len(nomm_users_emb)

9759

In [9]:
len(users_emb[325785555]), len(tracks_emb[115806632])

(1024, 256)

In [10]:
len(nomm_users_emb[325785555]), len(nomm_tracks_emb[115806632])

(768, 256)

In [11]:
Ue_mean = np.array(list(users_emb.values())).mean(axis=0)
Ge_mean = np.array(list(tracks_emb.values())).mean(axis=0)
Ue_mean.shape, Ge_mean.shape

((1024,), (256,))

In [12]:
Ue_nomm_mean = np.array(list(nomm_users_emb.values())).mean(axis=0)
Ge_nomm_mean = np.array(list(nomm_tracks_emb.values())).mean(axis=0)
Ue_nomm_mean.shape, Ge_nomm_mean.shape

((768,), (256,))

In [13]:
R = np.zeros((len(user2track2score_clean), len(track_wl_map)))
vs = list()

Ue = list()
Ge = [[]] * len(track_wl_map)

Ue_nomm = list()
Ge_nomm = [[]] * len(track_wl_map)


for track_id in track_wl_map:
    Ge[track_wl_map[track_id]] = tracks_emb.get(track_id, Ge_mean)
    Ge_nomm[track_wl_map[track_id]] = tracks_emb.get(track_id, Ge_nomm_mean)

for user_id, real_user_id in enumerate(user2track2score_clean):
    Ue.append(users_emb.get(real_user_id, Ue_mean))
    Ue_nomm.append(users_emb.get(real_user_id, Ue_nomm_mean))
    
    for track_id, score in user2track2score[real_user_id].items():
        if track_id in track_wl_map:
            R[user_id, track_wl_map[track_id]] = score
            
            vs.append(score)
            
Ue = np.array(Ue)
Ge = np.array(Ge)
            
Ue_nomm = np.array(Ue_nomm)
Ge_nomm = np.array(Ge_nomm)

for i in range(R.shape[0]):
    R[i][R[i] == 0] = np.min(R[i][R[i] != 0])
R

array([[ 0.26247394,  0.26247394,  2.28163106, ...,  0.26247394,
         2.19785737,  2.10689082],
       [ 0.05558677, -1.63433969, -1.63433969, ..., -1.63433969,
        -1.63433969, -1.63433969],
       [ 0.09070459,  0.09070459,  1.68303014, ...,  0.09070459,
         0.09070459,  1.67687085],
       ...,
       [-0.7106882 , -1.59261575, -0.62269729, ..., -1.59261575,
        -1.59261575, -1.59261575],
       [ 5.06037174,  3.04882283,  5.19142489, ...,  3.04882283,
         3.04882283,  5.09818687],
       [ 2.28674911,  0.47590584,  2.37860108, ...,  0.47590584,
         2.22571627,  0.47590584]])

In [14]:
import gc
del user2track2score_clean
del user2track2score
gc.collect()

0

In [15]:
R.shape, Ue.shape, Ge.shape

((10179, 8950), (10179, 1024), (8950, 256))

In [16]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-10-07 13:38:09.144261: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-07 13:38:09.172738: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-07 13:38:09.291397: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-10-07 13:38:09.292018: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-07 13:38:09.848502: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

matrix_approx_zeshel.py


# Open Data loader & context

In [17]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [18]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [19]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [20]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [21]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [22]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [23]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [24]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [25]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [26]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [27]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [28]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [29]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [30]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [31]:
with open("rbe_music_10179_8950_min.npz", "wb") as f:
    np.savez(f, R)  

In [32]:
R.shape

(10179, 8950)

In [33]:
ctx = EvalContextRelevs(
    R,
    det_attempts=0
)

Best det =  1.165097952016909e+63
Current de =  1.165097952016909e+63
100 7024 3054


array([3503, 4732, 2029, 4716, 5147, 6988, 5632, 8669, 6908,  228,  537,
       5752, 6324, 5332, 3223, 5406, 3997, 6793, 4885, 1522, 8026, 1541,
       4360, 5321, 7366, 5842, 8308, 4112, 1840, 7162,  701, 7998, 3313,
       4985, 4457, 2986, 2702, 2717, 6346, 8001, 2421, 8466, 2445, 5955,
       8016, 3881, 6051, 3976, 4267, 8228, 7740, 4997, 5299,  756, 8138,
       1589, 7288, 8793, 2006, 2833, 7213, 6161,  215, 3151, 7688, 2275,
       8472, 6160, 2830, 1444,  566, 2933, 1627, 4151, 5926, 5080,    0,
        143, 6221,  478, 6915, 6792, 1028,  522, 4687, 2660, 5621, 5663,
       6195, 2603, 7440, 5139, 6745, 8745,  843, 1583, 2617, 5687,  264,
       1407])

In [90]:
def run(i):
    np.random.seed(i)
    key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)
    ma = AnnCUR(ctx, key_games=key_games)
    return ma.get_score("test")

In [91]:
runs = [run(i) for i in range(15)]

np.mean(results), mse, len(results) =  0.15684020956123118 0.4547452544662179 3054
np.mean(results), mse, len(results) =  0.14315979043876884 0.4515905767442565 3054
np.mean(results), mse, len(results) =  0.14587426326129665 0.4533171082674798 3054
np.mean(results), mse, len(results) =  0.143958742632613 0.4493365121260629 3054
np.mean(results), mse, len(results) =  0.15296332678454486 0.45112237576265013 3054
np.mean(results), mse, len(results) =  0.1458218729535036 0.44901284763253496 3054
np.mean(results), mse, len(results) =  0.1500392927308448 0.44953141578461514 3054
np.mean(results), mse, len(results) =  0.1594400785854617 0.4516579007141788 3054
np.mean(results), mse, len(results) =  0.13853307138179438 0.4518306371820177 3054
np.mean(results), mse, len(results) =  0.16273084479371316 0.45013150153537584 3054
np.mean(results), mse, len(results) =  0.1531827111984283 0.45045650631417605 3054
np.mean(results), mse, len(results) =  0.1466371971185331 0.44885580401794967 3054
np.me

In [92]:
np.mean(runs), np.std(runs) / np.

(0.14921174416066363, 0.006646074742348704)

# pre-selected keys (just interesting)

In [48]:
import json
# !cat yt___home_ads_users_shevkunov_music_rbe_keys_v0_300
analytics_selected_key = list()
with open("yt___home_ads_users_shevkunov_music_rbe_keys_v0_300", "r") as f:
    for line in f:
        analytics_selected_key.append(
            track_wl_map[json.loads(line)["id"]]
        )
        # assert analytics_selected_key[-1] in track_wl
analytics_selected_key.sort()
len(analytics_selected_key)

300

In [50]:
mas = AnnCUR(ctx, key_games=analytics_selected_key)
mas.get_score("test")

np.mean(results), mse, len(results) =  0.18798297314996726 0.5342385150055138 3054


0.18798297314996726

In [52]:
key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)
ma = AnnCUR(ctx, key_games=key_games)
ma.get_score("test")

np.mean(results), mse, len(results) =  0.15730844793713164 0.4531827018130092 3054


0.15730844793713164

In [55]:
t = ctx.get_relevs("train").T
coitem_algorithm_key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])
ma = AnnCUR(ctx, key_games=coitem_algorithm_key_games)
ma.get_score("test")

/tmp/ipykernel_89641/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.14776031434184675 0.42510483229838064 3054


0.14776031434184675

In [57]:
ctx.key_games = analytics_selected_key
mas = AnnCUR(ctx)
mas.get_score("test")

np.mean(results), mse, len(results) =  0.18798297314996726 0.5342385150055138 3054


0.18798297314996726

In [ ]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (7024, 300)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (300, 300)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 300)
self.ge_users.shape =  (7024, 300)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0372 tf.Tensor(14.754914497869558, shape=(), dtype=float64) 100
slice = key, score = 0.0372
np.mean(results), mse, len(results) =  0.030667710706150344 tf.Tensor(15.953066916496029, shape=(), dtype=float64) 7024
slice = train, score = 0.030667710706150344
np.mean(results), mse, len(results) =  0.036067452521283565 tf.Tensor(16.192188673926065, shape=(), dtype=float64) 3054
slice = test, score = 0.036067452521283565

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.20049999999999998 tf.Tensor(15085475.697046129, shape=(), dtype=float64) 100
slice = key, score = 0.20049999999999998
np.mean(results), mse, len(results) =  0.19382545558086559 tf.Tensor(16564470.5086

np.mean(results), mse, len(results) =  0.19334851936218678 tf.Tensor(528491127113.8525, shape=(), dtype=float64) 7024
slice = train, score = 0.19334851936218678
np.mean(results), mse, len(results) =  0.18988212180746564 tf.Tensor(506959861127.97076, shape=(), dtype=float64) 3054
slice = test, score = 0.18988212180746564

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.20040000000000002 tf.Tensor(483148396897.51324, shape=(), dtype=float64) 100
slice = key, score = 0.20040000000000002
np.mean(results), mse, len(results) =  0.18785592255125286 tf.Tensor(533436703298.4179, shape=(), dtype=float64) 7024
slice = train, score = 0.18785592255125286
np.mean(results), mse, len(results) =  0.1898657498362803 tf.Tensor(510747992963.9415, shape=(), dtype=float64) 3054
slice = test, score = 0.1898657498362803

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.20710000000000003 tf.Tensor(693945555520.5973, shape=(), dtype=float64) 100
slice = key, score = 0.2071000000

## data-leaked popular !!
since zeroes-filling

In [40]:
ma = Popular(ctx)  

ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.2008129271070615 6.167076595260372 7024
np.mean(results), mse, len(results) =  0.19777668631303208 6.847915208206307 3054


(0.2008129271070615, 0.19777668631303208)

In [79]:
## no data-leaked popular !!

In [68]:
track_id_2_score = [list() for _ in range(len(track_wl_map))]

for user_id, real_user_id in enumerate(user2track2score_clean):
    for track_id, score in user2track2score[real_user_id].items():
        if track_id in track_wl_map:
            track_id_2_score[track_wl_map[track_id]].append(score)
            
preds = [0] * len(track_wl_map)

for k in range(len(track_id_2_score)):
    track_id_2_score[k] = np.mean(track_id_2_score[k])

In [69]:
track_id_2_score

[-0.28895759637706997,
 -0.3179078209387733,
 -0.23554237942254497,
 -0.8390913401763515,
 -0.4119836916370588,
 -0.3962638550438296,
 -0.34657961408808285,
 -0.4637839468595319,
 -0.2161860285212223,
 -0.37909952586281387,
 -0.2961676571115389,
 -0.23498849979624195,
 -0.3261616295623504,
 0.07878348502035643,
 -0.1792241496292472,
 -0.1563458338049198,
 -0.21694663688695473,
 -0.3943390550280146,
 -0.2695004674539376,
 -0.23702341805162314,
 0.13260054131792756,
 0.16514692413612614,
 -0.18236141789699348,
 -0.2898248561496854,
 0.07963795916734741,
 -0.2880945300598831,
 -0.26251991872582675,
 0.010695547202211596,
 -0.2277184615868018,
 -0.4343472355845436,
 -0.40224925381814186,
 -0.21851724723833788,
 -0.31312402925894367,
 -0.268288285415292,
 -0.2044354321649965,
 -1.1942528358030235,
 -0.08287101717378415,
 -0.052210201143479004,
 -0.9454656765611771,
 -0.8023102525790407,
 -0.40816864201264547,
 -0.15727856513404062,
 -0.1663142036463631,
 -0.3943305350946068,
 0.119762811004

In [64]:
track_id_2_score

defaultdict(list,
            {1081: -1.16804926352278,
             2635: -1.0226624913882503,
             2746: -1.0338455043237347,
             5123: -0.8922058784014458,
             5125: -0.9667549673969089,
             5126: -0.8113880270888832,
             6091: -0.9096425553160531,
             481: -0.9271975139307457,
             996: -1.4588658513849122,
             1198: -1.0302465077221836,
             1241: -1.3160008708242668,
             2778: -1.1294109407336752,
             4376: -0.9177921871565015,
             5618: -1.0838692254983173,
             139: -0.8351107342180837,
             2110: -0.993986542336135,
             7698: -1.3093514722160782,
             8344: -1.059988893248458,
             645: -1.0637101700883944,
             5887: -0.9678397150094308,
             1726: -0.8391009035439214,
             3208: -1.0502331184281675,
             4006: -0.8986190971339012,
             4012: -0.8370224124369554,
             4715: -1.04634234

In [74]:
t = "test"
rec_top = 100
tru_top = 100

recs = [track_id_2_score for _ in range(3054)]

if isinstance(recs, list):
    recs = np.array(recs)

if isinstance(recs, tf.Tensor):
    recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
else:
    recs = np.argsort(-recs, axis=1)[:, :rec_top]

trus = ma.get_user_scores(t)
trus = np.argsort(-trus, axis=1)[:, :tru_top]

ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

results = [
    ev(rec, tru) / float(tru_top)
    for rec, tru in zip(recs,trus)
]

print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 100, tru_top = 100, np.mean(results)=0.014181401440733465


In [80]:
np.argsort(-np.array(track_id_2_score))[:rec_top]

array([7458, 3873, 1995, 7293, 5740, 3391, 4107, 6037, 4041, 3282, 8819,
       8821, 4526, 5737, 8418, 6198, 2328, 1785, 6245, 3120, 3528, 6543,
       7470, 1376, 5742, 7010, 1397, 2227, 7825, 8488, 2639, 6709, 4200,
       1057, 8481, 1062, 5743, 1611,  352,  592, 6404, 2730,  997, 5736,
       3889, 2184, 5341,  926, 1873, 4228, 2496, 4449, 4506, 2425, 5375,
       8895,  729, 6163, 7127, 3000,  732, 5013, 3070, 6858, 3677, 5393,
       7542,  290, 2206, 7679, 7063, 6894,   21,  841, 8356, 1582, 6650,
       7463, 1215, 3415, 8668, 4179,  114, 8161, 5963, 8926,  765, 4524,
        437, 1581, 8652,  163, 6661, 1332, 3904, 6811,  976,  438, 1056,
        519])

In [77]:
track_id_2_score[7458], track_id_2_score[3873]

(0.4272803013154736, 0.39506575933754445)

In [81]:
ma = AnnCUR(ctx, key_games=np.argsort(-np.array(track_id_2_score))[:rec_top])

# (0.14708428246013666, 0.1577668631303209)
ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.15057517084282462 0.626038790303264 7024
np.mean(results), mse, len(results) =  0.1422167648984938 0.6165834370015782 3054


(0.15057517084282462, 0.1422167648984938)

In [42]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

# (0.14708428246013666, 0.1577668631303209)
ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_145060/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.1388240318906606 0.44161752774449825 7024
np.mean(results), mse, len(results) =  0.14776031434184675 0.42510483229838064 3054


(0.1388240318906606, 0.14776031434184675)

In [43]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f5a2f3b5db0>
np.mean(results), mse, len(results) =  0.2008129271070615 6.167076595260372 7024
np.mean(results), mse, len(results) =  0.19777668631303208 6.847915208206307 3054
0.2008129271070615 0.19777668631303208



model =  AnnCur(140027064006592)
np.mean(results), mse, len(results) =  0.1388240318906606 0.44161752774449825 7024
np.mean(results), mse, len(results) =  0.14776031434184675 0.42510483229838064 3054
0.1388240318906606 0.14776031434184675



model =  AnnCur(key_games=np.arange(100) - 140027064007600)
np.mean(results), mse, len(results) =  0.15983485193621869 0.4759125104639856 7024
np.mean(results), mse, len(results) =  0.15509168303863785 0.45566927826987635 3054
0.15983485193621869 0.15509168303863785



model =  AnnCur(random - 140027064006784)
np.mean(results), mse, len(results) =  0.1507858769931663 0.47102661274683705 7024
np.mean(results), mse, len(results) =  0.1546627373935822 0.45147063298356466 3054
0.1507858769931663 

In [41]:
class DssmKnnUGe(CBKnnV0):
    def __init__(self, ctx, ge, ue, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.ge = ge
        self.ue = ue
        
    def __repr__(self):
        return "DssmKnnUGe"
        
    def get_user_embs(self, t):
        return self.ue[t]
    
    def get_game_embs(self):
        return self.ge

In [45]:
Ue_ = {
    "key": Ue[:ctx.key_size][:, :256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, :256],
    "test": Ue[ctx.train_split:][:, :256], 
}

N = 1 # 1 # 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.20850000000000002 tf.Tensor(5.7390285311559355, shape=(), dtype=float64) 100
slice = key, score = 0.20850000000000002
np.mean(results), mse, len(results) =  0.20158171981776765 tf.Tensor(6.217564844332471, shape=(), dtype=float64) 7024
slice = train, score = 0.20158171981776765
np.mean(results), mse, len(results) =  0.1979993451211526 tf.Tensor(7.121708210124638, shape=(), dtype=float64) 3054
slice = test, score = 0.1979993451211526
last loss =  tf.Tensor(0.007468198761418495, shape=(), dtype=float64)
np.mean(results), mse, len(results) =  0.20158171981776765 tf.Tensor(6.217564844332471, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.1979993451211526 t

(0.20158171981776765, 0.1979993451211526)

In [46]:
select_head = 0
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 1 # 1 # 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.21050000000000002 tf.Tensor(6.830598047442549, shape=(), dtype=float64) 100
slice = key, score = 0.21050000000000002
np.mean(results), mse, len(results) =  0.20952733485193623 tf.Tensor(7.433643902708426, shape=(), dtype=float64) 7024
slice = train, score = 0.20952733485193623
np.mean(results), mse, len(results) =  0.18784872298624752 tf.Tensor(7.267022982548755, shape=(), dtype=float64) 3054
slice = test, score = 0.18784872298624752
last loss =  tf.Tensor(0.007562862908585589, shape=(), dtype=float64)
np.mean(results), mse, len(results) =  0.20952733485193623 tf.Tensor(7.433643902708426, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.18784872298624752

(0.20952733485193623, 0.18784872298624752)

In [47]:
select_head = 1
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 1 # 1 # 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.20500000000000004 tf.Tensor(7.007548068713604, shape=(), dtype=float64) 100
slice = key, score = 0.20500000000000004
np.mean(results), mse, len(results) =  0.20875711845102504 tf.Tensor(7.619624724669097, shape=(), dtype=float64) 7024
slice = train, score = 0.20875711845102504
np.mean(results), mse, len(results) =  0.18873935821872956 tf.Tensor(7.371512458137865, shape=(), dtype=float64) 3054
slice = test, score = 0.18873935821872956
last loss =  tf.Tensor(0.007557052881619865, shape=(), dtype=float64)
np.mean(results), mse, len(results) =  0.20875711845102504 tf.Tensor(7.619624724669097, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.18873935821872956

(0.20875711845102504, 0.18873935821872956)

In [48]:
select_head = 2
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 1 # 1 # 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0565 tf.Tensor(9.071528069530656, shape=(), dtype=float64) 100
slice = key, score = 0.0565
np.mean(results), mse, len(results) =  0.07670558086560365 tf.Tensor(9.797421221757117, shape=(), dtype=float64) 7024
slice = train, score = 0.07670558086560365
np.mean(results), mse, len(results) =  0.09626719056974462 tf.Tensor(9.015724085927818, shape=(), dtype=float64) 3054
slice = test, score = 0.09626719056974462
last loss =  tf.Tensor(0.0075654326646764376, shape=(), dtype=float64)
np.mean(results), mse, len(results) =  0.07670558086560365 tf.Tensor(9.797421221757117, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.09626719056974462 tf.Tensor(9.015724085927

(0.07670558086560365, 0.09626719056974462)

In [49]:
select_head = 3
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 1 # 1 # 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': False,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.1819 tf.Tensor(7.339206534027948, shape=(), dtype=float64) 100
slice = key, score = 0.1819
np.mean(results), mse, len(results) =  0.19571611617312074 tf.Tensor(7.955033928319079, shape=(), dtype=float64) 7024
slice = train, score = 0.19571611617312074
np.mean(results), mse, len(results) =  0.20053700065487887 tf.Tensor(7.6102476601153235, shape=(), dtype=float64) 3054
slice = test, score = 0.20053700065487887
last loss =  tf.Tensor(0.00756481468699169, shape=(), dtype=float64)
np.mean(results), mse, len(results) =  0.19571611617312074 tf.Tensor(7.955033928319079, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.20053700065487887 tf.Tensor(7.6102476601153

(0.19571611617312074, 0.20053700065487887)

In [54]:
d = dict()
for select_head in range(4):
    i = select_head
    Ue_ = {
        "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
        "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
        "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
    }

    N = 100000 #
    d[i] = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': False,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "Winit": "eye",
        "Wfreeze": True,
        "normalize_embs": False,
        "learning_rate": 0.001
    })
    d[i].fit()
    print('d[i].get_score("train"), d[i].get_score("test") = ', d[i].get_score("train"), d[i].get_score("test"))
    
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = size
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")
        
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = 100
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0955 tf.Tensor(6.7479440383829825, shape=(), dtype=float64) 100
slice = key, score = 0.0955
np.mean(results), mse, len(results) =  0.10784310933940774 tf.Tensor(7.38294183446751, shape=(), dtype=float64) 7024
slice = train, score = 0.10784310933940774
np.mean(results), mse, len(results) =  0.10115258677144728 tf.Tensor(7.236211327034941, shape=(), dtype=float64) 3054
slice = test, score = 0.10115258677144728

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.1017 tf.Tensor(90.85442264425025, shape=(), dtype=float64) 100
slice = key, score = 0.1017
np.mean(results), mse, len(results) =  0.13285734624145787 tf.Tensor(122.97868510120553, shape=(), dtype=float64) 70

np.mean(results), mse, len(results) =  0.2461133257403189 tf.Tensor(62238.42417929977, shape=(), dtype=float64) 7024
slice = train, score = 0.2461133257403189
np.mean(results), mse, len(results) =  0.23951211525867716 tf.Tensor(69339.4269614056, shape=(), dtype=float64) 3054
slice = test, score = 0.23951211525867716

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.2168 tf.Tensor(105842.21157313118, shape=(), dtype=float64) 100
slice = key, score = 0.2168
np.mean(results), mse, len(results) =  0.24496298405466968 tf.Tensor(77685.29086571487, shape=(), dtype=float64) 7024
slice = train, score = 0.24496298405466968
np.mean(results), mse, len(results) =  0.2394695481335953 tf.Tensor(83134.44781146536, shape=(), dtype=float64) 3054
slice = test, score = 0.2394695481335953

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.22870000000000004 tf.Tensor(81432.49759568191, shape=(), dtype=float64) 100
slice = key, score = 0.22870000000000004
np.mean(results), mse,


=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.24080000000000001 tf.Tensor(129027.46703929911, shape=(), dtype=float64) 100
slice = key, score = 0.24080000000000001
np.mean(results), mse, len(results) =  0.2617013097949886 tf.Tensor(103998.1147536468, shape=(), dtype=float64) 7024
slice = train, score = 0.2617013097949886
np.mean(results), mse, len(results) =  0.2519155206286837 tf.Tensor(119771.1499999356, shape=(), dtype=float64) 3054
slice = test, score = 0.2519155206286837

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.24810000000000007 tf.Tensor(71082.06905092139, shape=(), dtype=float64) 100
slice = key, score = 0.24810000000000007
np.mean(results), mse, len(results) =  0.2658086560364465 tf.Tensor(65138.960969911546, shape=(), dtype=float64) 7024
slice = train, score = 0.2658086560364465
np.mean(results), mse, len(results) =  0.25359528487229865 tf.Tensor(76211.90738906861, shape=(), dtype=float64) 3054
slice = test, score = 0.25359528487229


=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.25830000000000003 tf.Tensor(198137.36163965767, shape=(), dtype=float64) 100
slice = key, score = 0.25830000000000003
np.mean(results), mse, len(results) =  0.2789236902050114 tf.Tensor(223683.66477580686, shape=(), dtype=float64) 7024
slice = train, score = 0.2789236902050114
np.mean(results), mse, len(results) =  0.26527504911591354 tf.Tensor(186906.14231312802, shape=(), dtype=float64) 3054
slice = test, score = 0.26527504911591354

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.24849999999999997 tf.Tensor(232158.63616105984, shape=(), dtype=float64) 100
slice = key, score = 0.24849999999999997
np.mean(results), mse, len(results) =  0.27684937357630984 tf.Tensor(281949.1434782053, shape=(), dtype=float64) 7024
slice = train, score = 0.27684937357630984
np.mean(results), mse, len(results) =  0.2635297969875573 tf.Tensor(243795.65629772213, shape=(), dtype=float64) 3054
slice = test, score = 0.26352979


=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.27740000000000004 tf.Tensor(578200.4120247421, shape=(), dtype=float64) 100
slice = key, score = 0.27740000000000004
np.mean(results), mse, len(results) =  0.2986873576309796 tf.Tensor(552294.6701578728, shape=(), dtype=float64) 7024
slice = train, score = 0.2986873576309796
np.mean(results), mse, len(results) =  0.278860510805501 tf.Tensor(457993.85897118994, shape=(), dtype=float64) 3054
slice = test, score = 0.278860510805501

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.2765 tf.Tensor(622510.0758007314, shape=(), dtype=float64) 100
slice = key, score = 0.2765
np.mean(results), mse, len(results) =  0.30042283599088837 tf.Tensor(621283.8315203342, shape=(), dtype=float64) 7024
slice = train, score = 0.30042283599088837
np.mean(results), mse, len(results) =  0.2819351669941061 tf.Tensor(536084.9598270108, shape=(), dtype=float64) 3054
slice = test, score = 0.2819351669941061

=== Iteration 69000 ===


...

np.mean(results), mse, len(results) =  0.27418137813211846 tf.Tensor(70480.92142429651, shape=(), dtype=float64) 7024
slice = train, score = 0.27418137813211846
np.mean(results), mse, len(results) =  0.26150949574328747 tf.Tensor(98515.95869304461, shape=(), dtype=float64) 3054
slice = test, score = 0.26150949574328747

=== Iteration 27000 ===
np.mean(results), mse, len(results) =  0.25050000000000006 tf.Tensor(84790.39764020605, shape=(), dtype=float64) 100
slice = key, score = 0.25050000000000006
np.mean(results), mse, len(results) =  0.2741116173120729 tf.Tensor(63816.35963254188, shape=(), dtype=float64) 7024
slice = train, score = 0.2741116173120729
np.mean(results), mse, len(results) =  0.2612770137524558 tf.Tensor(90068.31029582612, shape=(), dtype=float64) 3054
slice = test, score = 0.2612770137524558

=== Iteration 28000 ===
np.mean(results), mse, len(results) =  0.2527 tf.Tensor(96462.1955442343, shape=(), dtype=float64) 100
slice = key, score = 0.2527
np.mean(results), mse, 

np.mean(results), mse, len(results) =  0.28977790432801825 tf.Tensor(144279.31925989914, shape=(), dtype=float64) 7024
slice = train, score = 0.28977790432801825
np.mean(results), mse, len(results) =  0.27283889980353637 tf.Tensor(124113.45560001842, shape=(), dtype=float64) 3054
slice = test, score = 0.27283889980353637

=== Iteration 44000 ===
np.mean(results), mse, len(results) =  0.2729 tf.Tensor(124842.50713218402, shape=(), dtype=float64) 100
slice = key, score = 0.2729
np.mean(results), mse, len(results) =  0.2955353075170843 tf.Tensor(176265.43896717782, shape=(), dtype=float64) 7024
slice = train, score = 0.2955353075170843
np.mean(results), mse, len(results) =  0.2777439423706614 tf.Tensor(140580.73788308149, shape=(), dtype=float64) 3054
slice = test, score = 0.2777439423706614

=== Iteration 45000 ===
np.mean(results), mse, len(results) =  0.268 tf.Tensor(129636.30254893703, shape=(), dtype=float64) 100
slice = key, score = 0.268
np.mean(results), mse, len(results) =  0.290

np.mean(results), mse, len(results) =  0.30018792710706144 tf.Tensor(334369.38030029455, shape=(), dtype=float64) 7024
slice = train, score = 0.30018792710706144
np.mean(results), mse, len(results) =  0.27525540275049115 tf.Tensor(292841.59107445064, shape=(), dtype=float64) 3054
slice = test, score = 0.27525540275049115

=== Iteration 61000 ===
np.mean(results), mse, len(results) =  0.2939 tf.Tensor(345361.7356493931, shape=(), dtype=float64) 100
slice = key, score = 0.2939
np.mean(results), mse, len(results) =  0.3112699316628701 tf.Tensor(396791.82055533765, shape=(), dtype=float64) 7024
slice = train, score = 0.3112699316628701
np.mean(results), mse, len(results) =  0.28703994760969226 tf.Tensor(343875.8988546754, shape=(), dtype=float64) 3054
slice = test, score = 0.28703994760969226

=== Iteration 62000 ===
np.mean(results), mse, len(results) =  0.2903 tf.Tensor(376829.068738047, shape=(), dtype=float64) 100
slice = key, score = 0.2903
np.mean(results), mse, len(results) =  0.310

np.mean(results), mse, len(results) =  0.3259111617312073 tf.Tensor(745158.9937234721, shape=(), dtype=float64) 7024
slice = train, score = 0.3259111617312073
np.mean(results), mse, len(results) =  0.29458087753765555 tf.Tensor(610776.1873947767, shape=(), dtype=float64) 3054
slice = test, score = 0.29458087753765555

=== Iteration 78000 ===
np.mean(results), mse, len(results) =  0.30539999999999995 tf.Tensor(850761.6211726003, shape=(), dtype=float64) 100
slice = key, score = 0.30539999999999995
np.mean(results), mse, len(results) =  0.32633826879271066 tf.Tensor(694887.3935452632, shape=(), dtype=float64) 7024
slice = train, score = 0.32633826879271066
np.mean(results), mse, len(results) =  0.2952619515389653 tf.Tensor(580899.7040653062, shape=(), dtype=float64) 3054
slice = test, score = 0.2952619515389653

=== Iteration 79000 ===
np.mean(results), mse, len(results) =  0.3074 tf.Tensor(1017956.9551435944, shape=(), dtype=float64) 100
slice = key, score = 0.3074
np.mean(results), mse

np.mean(results), mse, len(results) =  0.3345814350797267 tf.Tensor(1242163.0876405677, shape=(), dtype=float64) 7024
slice = train, score = 0.3345814350797267
np.mean(results), mse, len(results) =  0.2966699410609038 tf.Tensor(1121757.5074190088, shape=(), dtype=float64) 3054
slice = test, score = 0.2966699410609038

=== Iteration 95000 ===
np.mean(results), mse, len(results) =  0.3231 tf.Tensor(1033027.8595359524, shape=(), dtype=float64) 100
slice = key, score = 0.3231
np.mean(results), mse, len(results) =  0.33743308656036447 tf.Tensor(951836.8842542677, shape=(), dtype=float64) 7024
slice = train, score = 0.33743308656036447
np.mean(results), mse, len(results) =  0.2996987557301899 tf.Tensor(831496.9020974063, shape=(), dtype=float64) 3054
slice = test, score = 0.2996987557301899

=== Iteration 96000 ===
np.mean(results), mse, len(results) =  0.32270000000000004 tf.Tensor(1174594.1290301026, shape=(), dtype=float64) 100
slice = key, score = 0.32270000000000004
np.mean(results), ms

In [55]:
d_nomm = dict()
for select_head in range(3):
    i = select_head
    Ue_nomm = {
        "key": Ue_nomm[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
        "train": Ue_nomm[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
        "test": Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
    }

    N = 100000 #
    d[i] = DssmKnnUGe(ctx, Ge_nomm, Ue_nomm, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': False,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 5000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "Winit": "eye",
        "Wfreeze": True,
        "normalize_embs": False,
        "learning_rate": 0.001
    })
    d[i].fit()
    print('d[i].get_score("train"), d[i].get_score("test") = ', d[i].get_score("train"), d[i].get_score("test"))
    
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = size
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")
        
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = 100
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.09219999999999999 tf.Tensor(6.885874590375474, shape=(), dtype=float64) 100
slice = key, score = 0.09219999999999999
np.mean(results), mse, len(results) =  0.08336845102505695 tf.Tensor(7.470961321071042, shape=(), dtype=float64) 7024
slice = train, score = 0.08336845102505695
np.mean(results), mse, len(results) =  0.07501637197118534 tf.Tensor(7.273785087780422, shape=(), dtype=float64) 3054
slice = test, score = 0.07501637197118534

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.17410000000000003 tf.Tensor(9068.738240860583, shape=(), dtype=float64) 100
slice = key, score = 0.17410000000000003
np.mean(results), mse, len(results) =  0.19875854214123007 tf.Te

np.mean(results), mse, len(results) =  0.3107374715261959 tf.Tensor(1096065.9394737533, shape=(), dtype=float64) 7024
slice = train, score = 0.3107374715261959
np.mean(results), mse, len(results) =  0.28906679764243615 tf.Tensor(975945.3389274456, shape=(), dtype=float64) 3054
slice = test, score = 0.28906679764243615

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.29289999999999994 tf.Tensor(1380790.2227069552, shape=(), dtype=float64) 100
slice = key, score = 0.29289999999999994
np.mean(results), mse, len(results) =  0.30787727790432806 tf.Tensor(1278091.6404320158, shape=(), dtype=float64) 7024
slice = train, score = 0.30787727790432806
np.mean(results), mse, len(results) =  0.2852881466928618 tf.Tensor(1154227.8882356517, shape=(), dtype=float64) 3054
slice = test, score = 0.2852881466928618

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.29960000000000003 tf.Tensor(1430054.3611348618, shape=(), dtype=float64) 100
slice = key, score = 0.299600000

TypeError: unhashable type: 'slice'

In [68]:
d_nomm = dict()
for select_head in range(1, 3):
    i = select_head
    Ue_nomm_ = {
        "key": Ue_nomm[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
        "train": Ue_nomm[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
        "test": Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
    }

    N = 100000 #
    d[i] = DssmKnnUGe(ctx, Ge_nomm, Ue_nomm_, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': False,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 5000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "Winit": "eye",
        "Wfreeze": True,
        "normalize_embs": False,
        "learning_rate": 0.001
    })
    d[i].fit()
    print('d[i].get_score("train"), d[i].get_score("test") = ', d[i].get_score("train"), d[i].get_score("test"))
    
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = size
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")
        
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = 100
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.1159 tf.Tensor(6.91367650281996, shape=(), dtype=float64) 100
slice = key, score = 0.1159
np.mean(results), mse, len(results) =  0.114251138952164 tf.Tensor(7.528004394140682, shape=(), dtype=float64) 7024
slice = train, score = 0.114251138952164
np.mean(results), mse, len(results) =  0.10367714472822527 tf.Tensor(7.322075708781626, shape=(), dtype=float64) 3054
slice = test, score = 0.10367714472822527

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.18489999999999998 tf.Tensor(6295.349070227611, shape=(), dtype=float64) 100
slice = key, score = 0.18489999999999998
np.mean(results), mse, len(results) =  0.20491742596810936 tf.Tensor(4353.092221913133, shape=(

np.mean(results), mse, len(results) =  0.307000284738041 tf.Tensor(1312540.290788456, shape=(), dtype=float64) 7024
slice = train, score = 0.307000284738041
np.mean(results), mse, len(results) =  0.28323510150622133 tf.Tensor(1226342.2961356395, shape=(), dtype=float64) 3054
slice = test, score = 0.28323510150622133

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.2898 tf.Tensor(1339314.9102759135, shape=(), dtype=float64) 100
slice = key, score = 0.2898
np.mean(results), mse, len(results) =  0.30365746013667433 tf.Tensor(1674714.5014545084, shape=(), dtype=float64) 7024
slice = train, score = 0.30365746013667433
np.mean(results), mse, len(results) =  0.2777308447937132 tf.Tensor(1568498.5273794474, shape=(), dtype=float64) 3054
slice = test, score = 0.2777308447937132

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.3072 tf.Tensor(1385040.6675652377, shape=(), dtype=float64) 100
slice = key, score = 0.3072
np.mean(results), mse, len(results) =  0.3175

np.mean(results), mse, len(results) =  0.2587742027334852 tf.Tensor(304918.6199786182, shape=(), dtype=float64) 7024
slice = train, score = 0.2587742027334852
np.mean(results), mse, len(results) =  0.24737393582187295 tf.Tensor(239570.47158407696, shape=(), dtype=float64) 3054
slice = test, score = 0.24737393582187295

=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.2547 tf.Tensor(375538.5460620538, shape=(), dtype=float64) 100
slice = key, score = 0.2547
np.mean(results), mse, len(results) =  0.2701822323462415 tf.Tensor(392735.71294665575, shape=(), dtype=float64) 7024
slice = train, score = 0.2701822323462415
np.mean(results), mse, len(results) =  0.25761624099541586 tf.Tensor(279981.0403829434, shape=(), dtype=float64) 3054
slice = test, score = 0.25761624099541586

=== Iteration 55000 ===
np.mean(results), mse, len(results) =  0.25120000000000003 tf.Tensor(537733.5128465017, shape=(), dtype=float64) 100
slice = key, score = 0.25120000000000003
np.mean(results), ms

In [93]:
from sklearn.linear_model import Ridge

Z = "test"
R_All = ctx.get_relevs(Z)

select_head = 1
GE, QE = Ge_nomm, Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256]

GE_ = d[select_head].get_game_embs()
GE = np.hstack([
    GE_,
    d[select_head].g_dssm(GE_),
    # d[select_head].vb.numpy().reshape((-1, 1)) ,  # Vertical bias
    ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
])

R_All = ctx.get_relevs(t)
QE_ = d[select_head].get_user_embs(t)
QE = np.hstack([
    QE_ @ d[select_head].W,
    d[select_head].u_dssm(QE_),
    # np.ones((R_All.shape[0], 1)),
    np.ones((R_All.shape[0], 1)) * m.pb
])

best = 0
best_arg = None

STRIP = 0.05

MOMENTUM=0

def get_Rp_last(GE, A, R, QE_i, L, u_last, m):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    u = u * L + QE_i * (1 - L)
    Rp = GE @ u
    
    return Rp, u

for randomFirst in [False]:
    for K in [25]:
        for L in np.linspace(0, 1, 11):
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                    
best_arg

100%|█████████████████████████████████████████| 152/152 [00:53<00:00,  2.83it/s]


K = 25, L = 0.0, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.37407894736842107


100%|█████████████████████████████████████████| 152/152 [00:59<00:00,  2.54it/s]


K = 25, L = 0.1, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.37394736842105264


100%|█████████████████████████████████████████| 152/152 [00:54<00:00,  2.81it/s]


...

100%|█████████████████████████████████████████| 152/152 [00:58<00:00,  2.62it/s]


K = 25, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.376578947368421


100%|█████████████████████████████████████████| 152/152 [00:59<00:00,  2.57it/s]

K = 25, L = 1.0, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.2574342105263158


'K = 25, L = 0.9, rndFirst = False np.mean(results) = 0.376578947368421'

In [97]:
STRIP = 1

for randomFirst in [False]:
    for K in [25]:
        for L in [best_arg_d["L"]]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|███████████████████████████████████████| 3054/3054 [18:34<00:00,  2.74it/s]

K = 25, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.36328749181401443


'K = 25, L = 0.9, rndFirst = False np.mean(results) = 0.36328749181401443'

In [95]:
from sklearn.linear_model import Ridge

Z = "test"
R_All = ctx.get_relevs(Z)

select_head = 2
GE, QE = Ge_nomm, Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256]

GE_ = d[select_head].get_game_embs()
GE = np.hstack([
    GE_,
    d[select_head].g_dssm(GE_),
    # d[select_head].vb.numpy().reshape((-1, 1)) ,  # Vertical bias
    ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
])

R_All = ctx.get_relevs(t)
QE_ = d[select_head].get_user_embs(t)
QE = np.hstack([
    QE_ @ d[select_head].W,
    d[select_head].u_dssm(QE_),
    # np.ones((R_All.shape[0], 1)),
    np.ones((R_All.shape[0], 1)) * m.pb
])

best = 0
best_arg = None

STRIP = 0.05

MOMENTUM=0

def get_Rp_last(GE, A, R, QE_i, L, u_last, m):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    u = u * L + QE_i * (1 - L)
    Rp = GE @ u
    
    return Rp, u

for randomFirst in [False]:
    for K in [10]:
        for L in np.linspace(0, 1, 11):
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 152/152 [02:45<00:00,  1.09s/it]


K = 10, L = 0.0, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.35092105263157897


100%|█████████████████████████████████████████| 152/152 [02:35<00:00,  1.02s/it]


K = 10, L = 0.1, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.3511842105263158


100%|█████████████████████████████████████████| 152/152 [02:32<00:00,  1.01s/it]


...

100%|█████████████████████████████████████████| 152/152 [02:52<00:00,  1.14s/it]


K = 10, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.3628289473684211


100%|█████████████████████████████████████████| 152/152 [03:01<00:00,  1.20s/it]

K = 10, L = 1.0, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.2864473684210526


'K = 10, L = 0.9, rndFirst = False np.mean(results) = 0.3628289473684211'

In [96]:
STRIP = 1

for randomFirst in [False]:
    for K in [25]:
        for L in [best_arg_d["L"]]:
            for T, TS in [(200, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|███████████████████████████████████████| 3054/3054 [24:47<00:00,  2.05it/s]

K = 25, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.36328749181401443


'K = 25, L = 0.9, rndFirst = False np.mean(results) = 0.36328749181401443'

In [46]:
"""
select_head = 0
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0001
})
d.fit()
d.get_score("train"), d.get_score("test")
"""
pass
# works bad af

In [43]:
d = dict()
for select_head in range(1, 2):
    i = select_head
    Ue_nomm_ = {
        "key": Ue_nomm[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
        "train": Ue_nomm[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
        "test": Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
    }

    N = 100000 #
    d[i] = DssmKnnUGe(ctx, Ge_nomm, Ue_nomm_, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': False,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 5000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "Winit": "eye",
        "Wfreeze": True,
        "normalize_embs": False,
        "learning_rate": 0.001
    })
    d[i].fit()
    print('d[i].get_score("train"), d[i].get_score("test") = ', d[i].get_score("train"), d[i].get_score("test"))
    
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = size
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")
        
    for size in range(100, 1000, 100):
        t = "test"
        rec_top = size + 100
        tru_top = 100
        recs = d[i].recommend(t)

        if isinstance(recs, list):
            recs = np.array(recs)

        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :rec_top]

        trus = d[i].get_user_scores(t)
        trus = np.argsort(-trus, axis=1)[:, :tru_top]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        results = [
            ev(rec, tru) / float(tru_top)
            for rec, tru in zip(recs,trus)
        ]

        print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.1285 tf.Tensor(6.940146484940338, shape=(), dtype=float64) 100
slice = key, score = 0.1285
np.mean(results), mse, len(results) =  0.1274601366742597 tf.Tensor(7.516253663465308, shape=(), dtype=float64) 7024
slice = train, score = 0.1274601366742597
np.mean(results), mse, len(results) =  0.11837262606417813 tf.Tensor(7.30948051817114, shape=(), dtype=float64) 3054
slice = test, score = 0.11837262606417813

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.18 tf.Tensor(9096.493443025045, shape=(), dtype=float64) 100
slice = key, score = 0.18
np.mean(results), mse, len(results) =  0.2049245444191344 tf.Tensor(8082.743922519202, shape=(), dtype=float64) 7024
slice 

np.mean(results), mse, len(results) =  0.3089236902050114 tf.Tensor(883320.7246354423, shape=(), dtype=float64) 7024
slice = train, score = 0.3089236902050114
np.mean(results), mse, len(results) =  0.2829535036018337 tf.Tensor(750707.0034563655, shape=(), dtype=float64) 3054
slice = test, score = 0.2829535036018337

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.2997 tf.Tensor(881102.0608046389, shape=(), dtype=float64) 100
slice = key, score = 0.2997
np.mean(results), mse, len(results) =  0.31741884965831435 tf.Tensor(901995.4319555528, shape=(), dtype=float64) 7024
slice = train, score = 0.31741884965831435
np.mean(results), mse, len(results) =  0.28984610347085793 tf.Tensor(784641.6634131577, shape=(), dtype=float64) 3054
slice = test, score = 0.28984610347085793

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.2889 tf.Tensor(854004.8436630447, shape=(), dtype=float64) 100
slice = key, score = 0.2889
np.mean(results), mse, len(results) =  0.3122537

In [48]:
from sklearn.linear_model import Ridge

Z = "test"
R_All = ctx.get_relevs(Z)

select_head = 1
GE, QE = Ge_nomm, Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256]

GE_ = d[select_head].get_game_embs()
GE = np.hstack([
    GE_,
    d[select_head].g_dssm(GE_),
    # d[select_head].vb.numpy().reshape((-1, 1)) ,  # Vertical bias
    ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
])

R_All = ctx.get_relevs(t)
QE_ = d[select_head].get_user_embs(t)
QE = np.hstack([
    QE_ @ d[select_head].W,
    d[select_head].u_dssm(QE_),
    # np.ones((R_All.shape[0], 1)),
    np.ones((R_All.shape[0], 1)) * d[select_head].pb
])

best = 0
best_arg = None

STRIP = 0.05

MOMENTUM=0

def get_Rp_last(GE, A, R, QE_i, L, u_last, m):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    u = u * L + QE_i * (1 - L)
    Rp = GE @ u
    
    return Rp, u

for randomFirst in [False]:
    for K in [10]:
        for L in [0.9]:
            for T, TS in [(x + 100, x) for x in range(100, 1000, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 152/152 [02:49<00:00,  1.11s/it]


K = 10, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.38434210526315793


100%|█████████████████████████████████████████| 152/152 [04:12<00:00,  1.66s/it]


K = 10, L = 0.9, T = 300, TS = 200, rndFirst = False np.mean(results) = 0.3333223684210526


100%|█████████████████████████████████████████| 152/152 [05:01<00:00,  1.99s/it]


...

100%|█████████████████████████████████████████| 152/152 [13:08<00:00,  5.19s/it]


K = 10, L = 0.9, T = 900, TS = 800, rndFirst = False np.mean(results) = 0.28550986842105264


100%|█████████████████████████████████████████| 152/152 [15:09<00:00,  5.98s/it]

K = 10, L = 0.9, T = 1000, TS = 900, rndFirst = False np.mean(results) = 0.2892909356725147


'K = 10, L = 0.9, rndFirst = False np.mean(results) = 0.38434210526315793'

In [49]:
T

1000

In [53]:
from sklearn.linear_model import Ridge

Z = "test"
R_All = ctx.get_relevs(Z)

select_head = 1
GE, QE = Ge_nomm, Ue_nomm[ctx.train_split:][:, select_head*256:(select_head + 1) * 256]

GE_ = d[select_head].get_game_embs()
GE = np.hstack([
    GE_,
    d[select_head].g_dssm(GE_),
    # d[select_head].vb.numpy().reshape((-1, 1)) ,  # Vertical bias
    ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
])

R_All = ctx.get_relevs(t)
QE_ = d[select_head].get_user_embs(t)
QE = np.hstack([
    QE_ @ d[select_head].W,
    d[select_head].u_dssm(QE_),
    # np.ones((R_All.shape[0], 1)),
    np.ones((R_All.shape[0], 1)) * d[select_head].pb
])

best = 0
best_arg = None

STRIP = 0.05

MOMENTUM=0

def get_Rp_last(GE, A, R, QE_i, L, u_last, m):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape

    if u_last is not None:
        u = (u * (1 - m) + u_last * m)

    u = u * L + QE_i * (1 - L)
    Rp = GE @ u
    
    return Rp, u

for randomFirst in [False]:
    for K in [10]:
        for L in [0.9]:
            for T, TS in [(x + 100, 100) for x in range(100, 1000, 100)]:
                results = list()


                As = list()
                for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                    A = (
                        ctx.key_games[:K]
                        if randomFirst else
                        (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                    )
                    Rt = R_All[i]
                    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
                    
                    u_last = QE[i]
                    for _ in range(T // K - 1):
                        Rp, u_last = get_Rp_last(GE, A, Rt[A], QE[i], L, u_last, m=MOMENTUM)
                        
                        An = list(A)
                        for ai in Rp.argsort()[::-1]:
                            if True and (ai not in An):
                                An.append(ai)
                            if len(An) == len(A) + K:
                                break
                        A = np.array(An)
                        
                    assert len(A) == T
                    As.append(A)

                            
                    A = sorted([
                        (-Rt[ai], ai)
                        for ai in A
                    ])[:T]
                    A = [ai for _, ai in A]

                    pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                    # print("len(pred_top), len(gt_top) = ", len(pred_top), len(gt_top))
                    results.append(ev(pred_top, gt_top) / float(TS))

                if np.mean(results) > best:
                    best = np.mean(results)
                    best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                    best_arg_d = {
                        "K": K,
                        "L": L,
                        "randomFirst": randomFirst,
                        "score": np.mean(results)
                    }
                print(f"K = {K}, L = {L}, T = {T}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}")
                
best_arg

100%|█████████████████████████████████████████| 152/152 [02:06<00:00,  1.20it/s]


K = 10, L = 0.9, T = 200, TS = 100, rndFirst = False np.mean(results) = 0.38434210526315793


100%|█████████████████████████████████████████| 152/152 [03:45<00:00,  1.49s/it]


K = 10, L = 0.9, T = 300, TS = 100, rndFirst = False np.mean(results) = 0.4296052631578947


100%|█████████████████████████████████████████| 152/152 [05:28<00:00,  2.16s/it]


...

100%|█████████████████████████████████████████| 152/152 [13:18<00:00,  5.26s/it]


K = 10, L = 0.9, T = 900, TS = 100, rndFirst = False np.mean(results) = 0.5484868421052632


100%|█████████████████████████████████████████| 152/152 [14:08<00:00,  5.58s/it]

K = 10, L = 0.9, T = 1000, TS = 100, rndFirst = False np.mean(results) = 0.5630263157894737


'K = 10, L = 0.9, rndFirst = False np.mean(results) = 0.5630263157894737'

In [54]:
ctx.set_kmeans_games_as_key()

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


In [56]:
ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.16419561503416855 0.4615127960828185 7024
np.mean(results), mse, len(results) =  0.1661329404060249 0.4433120029950892 3054


(0.16419561503416855, 0.1661329404060249)

In [55]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 5000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.08370000000000001 tf.Tensor(2.728771681906084, shape=(), dtype=float64) 100
slice = key, score = 0.08370000000000001
np.mean(results), mse, len(results) =  0.0827349088838269 tf.Tensor(2.732115988156717, shape=(), dtype=float64) 7024
slice = train, score = 0.0827349088838269
np.mean(results), mse, len(results) =  0.09481990831696138 tf.Tensor(2.7072670473785387, shape=(), dtype=float64) 3054
slice = test, score = 0.09481990831696138

=== Iteration 5000 ===
np.mean(results), mse, len(results) =  0.2991 tf.Tensor(8299.869777503052, shape=(), dtype=float64) 100
slice = key, score = 0.2991
np.mean(results), mse, len(results) =  0.3088297266514807 tf.Tensor(7846.677756181609, shap

np.mean(results), mse, len(results) =  0.3719646365422397 tf.Tensor(85854.35473022955, shape=(), dtype=float64) 3054
slice = test, score = 0.3719646365422397

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.3779 tf.Tensor(87093.47331208008, shape=(), dtype=float64) 100
slice = key, score = 0.3779
np.mean(results), mse, len(results) =  0.39759253986332577 tf.Tensor(73385.21309932912, shape=(), dtype=float64) 7024
slice = train, score = 0.39759253986332577
np.mean(results), mse, len(results) =  0.3715455140798952 tf.Tensor(73530.85386505655, shape=(), dtype=float64) 3054
slice = test, score = 0.3715455140798952

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.37810000000000005 tf.Tensor(95771.55658601981, shape=(), dtype=float64) 100
slice = key, score = 0.37810000000000005
np.mean(results), mse, len(results) =  0.3967041571753986 tf.Tensor(84817.34924767775, shape=(), dtype=float64) 7024
slice = train, score = 0.3967041571753986
np.mean(results), mse, l

(0.4004683940774488, 0.37294368041912246)

In [51]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8950, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8950)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0854 tf.Tensor(4.01886514219453, shape=(), dtype=float64) 100
slice = key, score = 0.0854
np.mean(results), mse, len(results) =  0.07613752847380412 tf.Tensor(3.9641712156465467, shape=(), dtype=float64) 7024
slice = train, score = 0.07613752847380412
np.mean(results), mse, len(results) =  0.08562213490504257 tf.Tensor(3.7872724075612187, shape=(), dtype=float64) 3054
slice = test, score = 0.08562213490504257

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.23050000000000004 tf.Tensor(2208.295759832379, shape=(), dtype=float64) 100
slice = key, score = 0.23050000000000004
np.mean(results), mse, len(results) =  0.22645358769931664 tf.Tensor(2096.10414839048, sh

np.mean(results), mse, len(results) =  0.3663083712984055 tf.Tensor(8762.701908733376, shape=(), dtype=float64) 7024
slice = train, score = 0.3663083712984055
np.mean(results), mse, len(results) =  0.36367387033398824 tf.Tensor(7565.527915203276, shape=(), dtype=float64) 3054
slice = test, score = 0.36367387033398824

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.3484999999999999 tf.Tensor(8918.443933176266, shape=(), dtype=float64) 100
slice = key, score = 0.3484999999999999
np.mean(results), mse, len(results) =  0.36546697038724374 tf.Tensor(9764.689651448189, shape=(), dtype=float64) 7024
slice = train, score = 0.36546697038724374
np.mean(results), mse, len(results) =  0.3621840209561232 tf.Tensor(8427.47220698363, shape=(), dtype=float64) 3054
slice = test, score = 0.3621840209561232

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.35050000000000003 tf.Tensor(8542.735433438289, shape=(), dtype=float64) 100
slice = key, score = 0.35050000000000003


=== Iteration 33000 ===
np.mean(results), mse, len(results) =  0.3704 tf.Tensor(22752.116203603215, shape=(), dtype=float64) 100
slice = key, score = 0.3704
np.mean(results), mse, len(results) =  0.3898490888382688 tf.Tensor(21093.067282480442, shape=(), dtype=float64) 7024
slice = train, score = 0.3898490888382688
np.mean(results), mse, len(results) =  0.3827308447937131 tf.Tensor(18273.00414453949, shape=(), dtype=float64) 3054
slice = test, score = 0.3827308447937131

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.3754 tf.Tensor(22584.489361716413, shape=(), dtype=float64) 100
slice = key, score = 0.3754
np.mean(results), mse, len(results) =  0.3953402619589977 tf.Tensor(20208.49409514581, shape=(), dtype=float64) 7024
slice = train, score = 0.3953402619589977
np.mean(results), mse, len(results) =  0.38744269810085136 tf.Tensor(17871.34738819778, shape=(), dtype=float64) 3054
slice = test, score = 0.38744269810085136

=== Iteration 35000 ===
np.mean(results), mse,


=== Iteration 50000 ===
np.mean(results), mse, len(results) =  0.39769999999999994 tf.Tensor(31371.16983626701, shape=(), dtype=float64) 100
slice = key, score = 0.39769999999999994
np.mean(results), mse, len(results) =  0.4050583712984055 tf.Tensor(27355.565523807756, shape=(), dtype=float64) 7024
slice = train, score = 0.4050583712984055
np.mean(results), mse, len(results) =  0.3897085789129011 tf.Tensor(23524.54926193574, shape=(), dtype=float64) 3054
slice = test, score = 0.3897085789129011

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.38909999999999995 tf.Tensor(29577.116129009355, shape=(), dtype=float64) 100
slice = key, score = 0.38909999999999995
np.mean(results), mse, len(results) =  0.40563211845102504 tf.Tensor(25361.768193245687, shape=(), dtype=float64) 7024
slice = train, score = 0.40563211845102504
np.mean(results), mse, len(results) =  0.3917092337917485 tf.Tensor(21548.99773368488, shape=(), dtype=float64) 3054
slice = test, score = 0.391709233791

np.mean(results), mse, len(results) =  0.39098886705959396 tf.Tensor(25988.73454261333, shape=(), dtype=float64) 3054
slice = test, score = 0.39098886705959396

=== Iteration 67000 ===
np.mean(results), mse, len(results) =  0.39720000000000005 tf.Tensor(34757.9882504999, shape=(), dtype=float64) 100
slice = key, score = 0.39720000000000005
np.mean(results), mse, len(results) =  0.4136090546697039 tf.Tensor(30521.83270835934, shape=(), dtype=float64) 7024
slice = train, score = 0.4136090546697039
np.mean(results), mse, len(results) =  0.39587753765553374 tf.Tensor(25977.823249323672, shape=(), dtype=float64) 3054
slice = test, score = 0.39587753765553374

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.3957 tf.Tensor(39525.110636712656, shape=(), dtype=float64) 100
slice = key, score = 0.3957
np.mean(results), mse, len(results) =  0.41288866742596814 tf.Tensor(35078.99188148938, shape=(), dtype=float64) 7024
slice = train, score = 0.41288866742596814
np.mean(results), m

np.mean(results), mse, len(results) =  0.41717824601366743 tf.Tensor(43822.23391568347, shape=(), dtype=float64) 7024
slice = train, score = 0.41717824601366743
np.mean(results), mse, len(results) =  0.39676489849377866 tf.Tensor(37811.17995288159, shape=(), dtype=float64) 3054
slice = test, score = 0.39676489849377866

=== Iteration 84000 ===
np.mean(results), mse, len(results) =  0.3894 tf.Tensor(50339.10817414428, shape=(), dtype=float64) 100
slice = key, score = 0.3894
np.mean(results), mse, len(results) =  0.4077676537585421 tf.Tensor(45780.398076471654, shape=(), dtype=float64) 7024
slice = train, score = 0.4077676537585421
np.mean(results), mse, len(results) =  0.3911820563195809 tf.Tensor(39335.85130310571, shape=(), dtype=float64) 3054
slice = test, score = 0.3911820563195809

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.3968 tf.Tensor(54566.18131416812, shape=(), dtype=float64) 100
slice = key, score = 0.3968
np.mean(results), mse, len(results) =  0.413883

np.mean(results), mse, len(results) =  0.41637670842824603 tf.Tensor(53589.83892971107, shape=(), dtype=float64) 7024
np.mean(results), mse, len(results) =  0.39644073346430914 tf.Tensor(47262.354747634534, shape=(), dtype=float64) 3054


(0.41637670842824603, 0.39644073346430914)

# Cross Scores

In [52]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size
    tru_top = size
    recs = m.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = m.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 100, tru_top = 100, np.mean(results)=0.39644073346430914
rec_top = 200, tru_top = 200, np.mean(results)=0.4470776031434185
rec_top = 300, tru_top = 300, np.mean(results)=0.4693123772102161
rec_top = 400, tru_top = 400, np.mean(results)=0.48327439423706614
rec_top = 500, tru_top = 500, np.mean(results)=0.49288605108055006
rec_top = 600, tru_top = 600, np.mean(results)=0.5008387906570617
rec_top = 700, tru_top = 700, np.mean(results)=0.5069548133595285
rec_top = 800, tru_top = 800, np.mean(results)=0.5118852324819909
rec_top = 900, tru_top = 900, np.mean(results)=0.516167503456305


In [53]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size
    tru_top = 100
    recs = m.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = m.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 100, tru_top = 100, np.mean(results)=0.39644073346430914
rec_top = 200, tru_top = 100, np.mean(results)=0.5435166994106091
rec_top = 300, tru_top = 100, np.mean(results)=0.6252750491159136
rec_top = 400, tru_top = 100, np.mean(results)=0.6819220694171578
rec_top = 500, tru_top = 100, np.mean(results)=0.7252685003274394
rec_top = 600, tru_top = 100, np.mean(results)=0.759947609692207
rec_top = 700, tru_top = 100, np.mean(results)=0.788814669286182
rec_top = 800, tru_top = 100, np.mean(results)=0.8135199738048462
rec_top = 900, tru_top = 100, np.mean(results)=0.8343418467583497


In [55]:
select_head = 0
Ue_ = {
    "key": Ue[:ctx.key_size][:, select_head*256:(select_head + 1) * 256],
    "train": Ue[ctx.key_size + 1: ctx.train_split][:, select_head*256:(select_head + 1) * 256],
    "test": Ue[ctx.train_split:][:, select_head*256:(select_head + 1) * 256], 
}

N = 100000 #
d = DssmKnnUGe(ctx, Ge, Ue_, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.001
})
d.fit()
d.get_score("train"), d.get_score("test")

self.embed_users['train'].shape =  (7024, 100)
self.embed_games.shape =  (8127, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8127)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (7024, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0774 tf.Tensor(6.797846491718012, shape=(), dtype=float64) 100
slice = key, score = 0.0774
np.mean(results), mse, len(results) =  0.09051822323462416 tf.Tensor(7.4184610585312, shape=(), dtype=float64) 7024
slice = train, score = 0.09051822323462416
np.mean(results), mse, len(results) =  0.07758022265880812 tf.Tensor(7.287030658766582, shape=(), dtype=float64) 3054
slice = test, score = 0.07758022265880812

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.04050000000000001 tf.Tensor(12157.161361836539, shape=(), dtype=float64) 100
slice = key, score = 0.04050000000000001
np.mean(results), mse, len(results) =  0.07976793849658313 tf.Tensor(12878.130607671988, sh

np.mean(results), mse, len(results) =  0.23914436218678817 tf.Tensor(40932.57374278028, shape=(), dtype=float64) 7024
slice = train, score = 0.23914436218678817
np.mean(results), mse, len(results) =  0.22835297969875573 tf.Tensor(49199.272661382434, shape=(), dtype=float64) 3054
slice = test, score = 0.22835297969875573

=== Iteration 17000 ===
np.mean(results), mse, len(results) =  0.2291 tf.Tensor(49199.7540203394, shape=(), dtype=float64) 100
slice = key, score = 0.2291
np.mean(results), mse, len(results) =  0.24684510250569477 tf.Tensor(42378.64985968766, shape=(), dtype=float64) 7024
slice = train, score = 0.24684510250569477
np.mean(results), mse, len(results) =  0.23469220694171578 tf.Tensor(52200.38138279889, shape=(), dtype=float64) 3054
slice = test, score = 0.23469220694171578

=== Iteration 18000 ===
np.mean(results), mse, len(results) =  0.22699999999999998 tf.Tensor(51456.54250398624, shape=(), dtype=float64) 100
slice = key, score = 0.22699999999999998
np.mean(results), 

np.mean(results), mse, len(results) =  0.2680139521640091 tf.Tensor(85545.71635907446, shape=(), dtype=float64) 7024
slice = train, score = 0.2680139521640091
np.mean(results), mse, len(results) =  0.25258677144728225 tf.Tensor(101106.95238397577, shape=(), dtype=float64) 3054
slice = test, score = 0.25258677144728225

=== Iteration 34000 ===
np.mean(results), mse, len(results) =  0.2587 tf.Tensor(95314.31558864476, shape=(), dtype=float64) 100
slice = key, score = 0.2587
np.mean(results), mse, len(results) =  0.2702107061503417 tf.Tensor(82875.89021221385, shape=(), dtype=float64) 7024
slice = train, score = 0.2702107061503417
np.mean(results), mse, len(results) =  0.25366404715127705 tf.Tensor(95067.71675914145, shape=(), dtype=float64) 3054
slice = test, score = 0.25366404715127705

=== Iteration 35000 ===
np.mean(results), mse, len(results) =  0.25710000000000005 tf.Tensor(116721.67019115601, shape=(), dtype=float64) 100
slice = key, score = 0.25710000000000005
np.mean(results), ms

np.mean(results), mse, len(results) =  0.2825968109339408 tf.Tensor(177024.11590574615, shape=(), dtype=float64) 7024
slice = train, score = 0.2825968109339408
np.mean(results), mse, len(results) =  0.26379829731499677 tf.Tensor(160643.03945684189, shape=(), dtype=float64) 3054
slice = test, score = 0.26379829731499677

=== Iteration 51000 ===
np.mean(results), mse, len(results) =  0.27099999999999996 tf.Tensor(123543.76243389433, shape=(), dtype=float64) 100
slice = key, score = 0.27099999999999996
np.mean(results), mse, len(results) =  0.28290575170842824 tf.Tensor(162493.854863584, shape=(), dtype=float64) 7024
slice = train, score = 0.28290575170842824
np.mean(results), mse, len(results) =  0.2636509495743288 tf.Tensor(155369.95343424595, shape=(), dtype=float64) 3054
slice = test, score = 0.2636509495743288

=== Iteration 52000 ===
np.mean(results), mse, len(results) =  0.2748999999999999 tf.Tensor(126933.26323221783, shape=(), dtype=float64) 100
slice = key, score = 0.27489999999

np.mean(results), mse, len(results) =  0.29721810933940773 tf.Tensor(460182.1209362567, shape=(), dtype=float64) 7024
slice = train, score = 0.29721810933940773
np.mean(results), mse, len(results) =  0.2742370661427636 tf.Tensor(401510.2056507271, shape=(), dtype=float64) 3054
slice = test, score = 0.2742370661427636

=== Iteration 68000 ===
np.mean(results), mse, len(results) =  0.28819999999999996 tf.Tensor(258004.4766810896, shape=(), dtype=float64) 100
slice = key, score = 0.28819999999999996
np.mean(results), mse, len(results) =  0.29867454441913444 tf.Tensor(365972.2894379046, shape=(), dtype=float64) 7024
slice = train, score = 0.29867454441913444
np.mean(results), mse, len(results) =  0.27469548133595284 tf.Tensor(317091.0866978786, shape=(), dtype=float64) 3054
slice = test, score = 0.27469548133595284

=== Iteration 69000 ===
np.mean(results), mse, len(results) =  0.29339999999999994 tf.Tensor(294951.49999553995, shape=(), dtype=float64) 100
slice = key, score = 0.29339999999


=== Iteration 84000 ===
np.mean(results), mse, len(results) =  0.2927 tf.Tensor(678423.0693126796, shape=(), dtype=float64) 100
slice = key, score = 0.2927
np.mean(results), mse, len(results) =  0.2988582004555809 tf.Tensor(803141.4549742904, shape=(), dtype=float64) 7024
slice = train, score = 0.2988582004555809
np.mean(results), mse, len(results) =  0.27642108709888674 tf.Tensor(747452.7670756982, shape=(), dtype=float64) 3054
slice = test, score = 0.27642108709888674

=== Iteration 85000 ===
np.mean(results), mse, len(results) =  0.3068 tf.Tensor(422696.6709398441, shape=(), dtype=float64) 100
slice = key, score = 0.3068
np.mean(results), mse, len(results) =  0.31227363325740315 tf.Tensor(577697.2163709166, shape=(), dtype=float64) 7024
slice = train, score = 0.31227363325740315
np.mean(results), mse, len(results) =  0.28477734119187953 tf.Tensor(527997.902087936, shape=(), dtype=float64) 3054
slice = test, score = 0.28477734119187953

=== Iteration 86000 ===
np.mean(results), mse,

(0.3206036446469248, 0.28920759659463)

In [59]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size
    tru_top = size
    recs = d.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = d.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 100, tru_top = 100, np.mean(results)=0.28920759659463
rec_top = 200, tru_top = 200, np.mean(results)=0.2744269810085135
rec_top = 300, tru_top = 300, np.mean(results)=0.26265880812049774
rec_top = 400, tru_top = 400, np.mean(results)=0.2557907662082515
rec_top = 500, tru_top = 500, np.mean(results)=0.252297969875573
rec_top = 600, tru_top = 600, np.mean(results)=0.25207378301680855
rec_top = 700, tru_top = 700, np.mean(results)=0.25448171016933296
rec_top = 800, tru_top = 800, np.mean(results)=0.2586693680419122
rec_top = 900, tru_top = 900, np.mean(results)=0.2643385723641126


In [57]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size
    tru_top = 100
    recs = d.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = d.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 100, tru_top = 100, np.mean(results)=0.28920759659463
rec_top = 200, tru_top = 100, np.mean(results)=0.3712213490504257
rec_top = 300, tru_top = 100, np.mean(results)=0.4123706614276359
rec_top = 400, tru_top = 100, np.mean(results)=0.43907007203667325
rec_top = 500, tru_top = 100, np.mean(results)=0.4601768172888016
rec_top = 600, tru_top = 100, np.mean(results)=0.47831041257367385
rec_top = 700, tru_top = 100, np.mean(results)=0.4949574328749181
rec_top = 800, tru_top = 100, np.mean(results)=0.5106712508185987
rec_top = 900, tru_top = 100, np.mean(results)=0.5260019646365421


In [60]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size + 100
    tru_top = size
    recs = d.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = d.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 200, tru_top = 100, np.mean(results)=0.3712213490504257
rec_top = 300, tru_top = 200, np.mean(results)=0.3169449901768173
rec_top = 400, tru_top = 300, np.mean(results)=0.2923510150622135
rec_top = 500, tru_top = 400, np.mean(results)=0.27939751146037983
rec_top = 600, tru_top = 500, np.mean(results)=0.2730366732154551
rec_top = 700, tru_top = 600, np.mean(results)=0.27130757476533507
rec_top = 800, tru_top = 700, np.mean(results)=0.27269435868650016
rec_top = 900, tru_top = 800, np.mean(results)=0.2762585952848723
rec_top = 1000, tru_top = 900, np.mean(results)=0.2813825220112057


In [61]:
for size in range(100, 1000, 100):
    t = "test"
    rec_top = size + 100
    tru_top = 100
    recs = d.recommend(t)

    if isinstance(recs, list):
        recs = np.array(recs)

    if isinstance(recs, tf.Tensor):
        recs = tf.argsort(-recs, axis=1)[:, :rec_top].numpy()
    else:
        recs = np.argsort(-recs, axis=1)[:, :rec_top]

    trus = d.get_user_scores(t)
    trus = np.argsort(-trus, axis=1)[:, :tru_top]

    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    results = [
        ev(rec, tru) / float(tru_top)
        for rec, tru in zip(recs,trus)
    ]

    print(f"rec_top = {rec_top}, tru_top = {tru_top}, np.mean(results)={np.mean(results)}")

rec_top = 200, tru_top = 100, np.mean(results)=0.3712213490504257
rec_top = 300, tru_top = 100, np.mean(results)=0.4123706614276359
rec_top = 400, tru_top = 100, np.mean(results)=0.43907007203667325
rec_top = 500, tru_top = 100, np.mean(results)=0.4601768172888016
rec_top = 600, tru_top = 100, np.mean(results)=0.47831041257367385
rec_top = 700, tru_top = 100, np.mean(results)=0.4949574328749181
rec_top = 800, tru_top = 100, np.mean(results)=0.5106712508185987
rec_top = 900, tru_top = 100, np.mean(results)=0.5260019646365421
rec_top = 1000, tru_top = 100, np.mean(results)=0.5402979698755731
